# Chunks: from the diarization and the word alignments
Cuts every audio into chunks, using the RTTM from `04_diarization` and the word times from `forced_alignment.ipynb` (`data/alignments/`).

**Steps** (all on the timeline of the original audio)
1. Cut at every speaker change, in the middle of the silence (or overlap) between the two speakers.
   A pause longer than `MAX_PAUSE_SEC` becomes a chunk of its own (`speaker: null`); the speech on each side keeps `EDGE_SEC` of it.
2. A cut that falls inside a word moves to the nearer edge of that word, so words are not cut in half.
3. **Merge:** neighbouring chunks of the same speaker become one chunk, with the pauses between them.
4. **Sentences:** a chunk is also cut between two sentences (a word ending in `.` `?` `!` `…`) when the pause between them is longer than `SENTENCE_PAUSE_SEC`, in the middle of that pause.
5. A chunk whose wav would still be longer than `MAX_CHUNK_SEC` is split into as few pieces as possible, each cut at the longest pause where it can go.

**Nothing is dropped.** The chunks follow each other with no gaps and add up to the audio's length. There is no padding: each wav holds exactly its chunk, so the wavs add up to the audio's length too.

**Text.** Every aligned word goes to the chunk that holds its middle, so each word is in exactly one chunk, in order.
The word times come from forced alignment (accurate to about 80 ms; with no padding, a cut at a word edge has no margin, so an alignment error there can clip a word's first or last sound). What this can't fix: mistakes in the Whisper transcript itself.

**Output**, in `outputs/chunks/<name>/` (an earlier run's wavs, `manifest.jsonl` and `summary.json` there are deleted):
- `wavs/0000_speaker_0.wav`, … (mono, the original sample rate)
- `chunks.json`: `audio`, `duration`, and per chunk `file, speaker, start, end, duration, text`.
  `start`/`end` are the chunk in the original audio; `duration` is the wav's length (`end - start`).
  It is written last, so audios that have one are skipped unless `FORCE = True`.

In [13]:
import json
import math
import os
import traceback
from bisect import bisect_left, bisect_right
from pathlib import Path

import numpy as np
import soundfile as sf

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
out_root = root / "outputs"
words_dir = root / "data" / "alignments"
wavs = sorted((root / "data" / "raw_audios").glob("*.wav"))
for i, p in enumerate(wavs):
    print(i, p.name)

0 18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak.wav
1 2026-yilda Google'ga ishga kirish XARITASI.wav
2 28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili.wav
3 29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi.wav
4 31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda.wav
5 40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son.wav
6 50 yoshida ham studentlikni tanlaydigan odam bilan suhbat.wav
7 99% odamlar buni eplay olmaydi - Iqtibos podcast #27.wav
8 AQLI PASTlar avlodi (bo'lib qolyapmiz).wav
9 Afsus qilmaslik uchun bugun nima qilish kerak - Hasanxon Yahyo Abdulmajid.wav
10 Chegaralar torayayotganini his qildim - Jamshidxon Ziyoxonov.wav
11 Dasturchilik faoliyatimni yakunlayman! - Azim Pulat Gashtak podkastida.wav
12 Ertaga bankrot bo‘lmaslik uchun hozir nima qilish kerak - Alisher Isayev.wav
13 HOZIRGI YOSHLAR NIMADAN BOSHLASHI KERA

In [14]:
ONLY = None           # None = every audio; or a list of indices from the list above, e.g. [3, 5]
FORCE = False         # True = rebuild chunks that already have a chunks.json

MAX_PAUSE_SEC = 3.0   # a pause longer than this is a chunk of its own, even between turns of the same speaker
EDGE_SEC = 0.5        # at such a pause (and at the start / end of the audio) each side keeps this much of it
SENTENCE_PAUSE_SEC = 0.5  # cut between two sentences when the pause between them is longer than this (None = off)
MAX_CHUNK_SEC = 30.0  # longest wav; longer merged chunks are split at pauses
MIN_SPLIT_SEC = 2.0   # splitting a long chunk never leaves a piece shorter than this
SNAP_SEC = 0.3        # a cut inside a word moves to the word's nearer edge, if that is at most this far
FADE_SEC = 0.01       # fade in / out at the wav edges, against clicks

assert MAX_PAUSE_SEC > 2 * EDGE_SEC

## Cuts, merge, wavs

In [15]:
RES = 100                    # grid: frames per second
SILENCE, OVERLAP = -1, -2    # frame labels; 0, 1, 2, ... = index of the one speaker talking


def read_rttm(path):
    turns = []
    for line in path.read_text().splitlines():
        p = line.split()  # counted from the end, so an id with spaces still works
        s = float(p[-7])
        turns.append((s, s + float(p[-6]), p[-3]))
    return sorted(turns)


def frame_labels(turns, n):
    """speakers, active[speaker, frame], labels[frame] (speaker index / SILENCE / OVERLAP)."""
    speakers = sorted({spk for *_, spk in turns})
    active = np.zeros((len(speakers), n), dtype=bool)
    for s, e, spk in turns:
        active[speakers.index(spk), max(0, round(s * RES)):min(n, round(e * RES))] = True
    count = active.sum(axis=0)
    labels = np.full(n, SILENCE, dtype=np.int16)
    labels[count > 1] = OVERLAP
    if speakers:
        one = count == 1
        labels[one] = active[:, one].argmax(axis=0)
    return speakers, active, labels


def runs(labels):
    """(label, start, end) of every stretch of equal labels, in frames."""
    change = np.flatnonzero(np.diff(labels)) + 1
    starts, ends = np.r_[0, change], np.r_[change, len(labels)]
    return [(int(labels[a]), int(a), int(b)) for a, b in zip(starts, ends)]


def speaker_of(s, e, speakers, active):
    """The speaker who talks most in s..e (seconds), or None if nobody does."""
    talk = active[:, round(s * RES):round(e * RES)].sum(axis=1)
    return speakers[int(talk.argmax())] if len(talk) and talk.max() > 0 else None


def diarization_cuts(labels):
    """Step 1, in frames: a cut at every speaker change; a pause longer than MAX_PAUSE_SEC
    gets a cut on each side (EDGE_SEC from the speech) and becomes a chunk of its own."""
    n, edge, long_pause = len(labels), round(EDGE_SEC * RES), MAX_PAUSE_SEC * RES
    talk = [(lab, a, b) for lab, a, b in runs(labels) if lab >= 0]
    cuts = {0, n}
    if not talk:
        return sorted(cuts)
    if talk[0][1] > long_pause:       # silence before the first speaker
        cuts.add(talk[0][1] - edge)
    if n - talk[-1][2] > long_pause:  # and after the last one
        cuts.add(talk[-1][2] + edge)
    for (l1, _, b1), (l2, a2, _) in zip(talk, talk[1:]):
        gap = a2 - b1  # silence and / or overlap between the two
        if gap > long_pause and (labels[b1:a2] == SILENCE).all():
            cuts.update((b1 + edge, a2 - edge))
        elif l1 != l2:
            cuts.add((b1 + a2) // 2)
    return sorted(cuts)


def snap(t, starts, ends):
    """Step 2: a cut inside a word moves to the word's nearer edge (if that is at most SNAP_SEC away)."""
    i = int(np.searchsorted(starts, t, side="right")) - 1
    if i >= 0 and starts[i] < t < ends[i]:
        edge = starts[i] if t - starts[i] <= ends[i] - t else ends[i]
        if abs(edge - t) <= SNAP_SEC:
            return float(edge)
    return t


def merge_same_speaker(cuts, speakers, active):
    """Step 3: drop every cut that has the same speaker on both sides."""
    spk = [speaker_of(s, e, speakers, active) for s, e in zip(cuts, cuts[1:])]
    inner = [t for t, a, b in zip(cuts[1:-1], spk, spk[1:]) if a != b]
    return [cuts[0]] + inner + [cuts[-1]]


def ends_sentence(word):
    return word.rstrip("\"'»”’)]").endswith((".", "?", "!", "…"))


def sentence_cuts(cuts, words):
    """Step 4: the middle of every pause longer than SENTENCE_PAUSE_SEC after a word that ends a sentence.
    Not where a cut is already in that pause (or so close that the piece between them would hold no word)."""
    if SENTENCE_PAUSE_SEC is None:
        return []
    new = []
    for a, b in zip(words, words[1:]):
        if ends_sentence(a["word"]) and b["start"] - a["end"] > SENTENCE_PAUSE_SEC:
            t = (a["end"] + b["start"]) / 2
            k = bisect_right(cuts, t)  # cuts[k - 1] <= t < cuts[k]
            if cuts[k - 1] <= (a["start"] + a["end"]) / 2 and (b["start"] + b["end"]) / 2 < cuts[k]:
                new.append(t)  # both pieces keep their word
    return new


def pause_candidates(labels, starts, ends):
    """(time, length) of every pause, sorted by time: silences between turns and gaps between words."""
    cands = [((a + b) / 2 / RES, (b - a) / RES) for lab, a, b in runs(labels) if lab == SILENCE]
    cands += [(float(e + s) / 2, float(s - e)) for e, s in zip(ends[:-1], starts[1:])]
    return sorted(cands)


def split_points(s, e, cands, times, starts, ends):
    """Step 5: cuts that split s..e into as few pieces as possible, each short enough for a wav of at most
    MAX_CHUNK_SEC. Each cut is the longest pause in the window that keeps the number of pieces minimal
    (equal pauses: the one that makes the pieces most even). No pause there: the even point."""
    limit = MAX_CHUNK_SEC - SNAP_SEC  # snapping can move a cut by up to SNAP_SEC: the wav stays <= MAX_CHUNK_SEC
    if e - s <= limit:
        return []
    k = math.ceil((e - s) / limit)   # pieces needed
    even = s + (e - s) / k
    lo = bisect_left(times, max(s + MIN_SPLIT_SEC, e - (k - 1) * limit))
    hi = bisect_right(times, min(s + limit, e - MIN_SPLIT_SEC))
    t = even
    if lo < hi:
        t = max(cands[lo:hi], key=lambda c: (round(c[1], 1), -abs(c[0] - even)))[0]
    t = snap(t, starts, ends)
    if not s < t < e:
        t = even
    return [t] + split_points(t, e, cands, times, starts, ends)


def chunk_bounds(labels, total_sec, words, speakers, active):
    """Steps 1-5. Sorted times from 0 to total_sec: chunk k is bounds[k]..bounds[k + 1]."""
    starts = np.array([w["start"] for w in words], dtype=float)
    ends = np.array([w["end"] for w in words], dtype=float)
    cuts = [c / RES for c in diarization_cuts(labels)][1:-1]
    cuts = [0.0] + [snap(t, starts, ends) for t in cuts] + [total_sec]
    cuts = sorted({round(float(t), 3) for t in cuts if 0 <= t <= total_sec})
    cuts = merge_same_speaker(cuts, speakers, active)
    cuts = sorted({round(float(t), 3) for t in cuts + sentence_cuts(cuts, words)})

    cands = pause_candidates(labels, starts, ends)
    times = [t for t, _ in cands]
    bounds = []
    for s, e in zip(cuts, cuts[1:]):
        bounds += [s] + split_points(s, e, cands, times, starts, ends)
    return sorted({round(float(t), 3) for t in bounds + [total_sec] if 0 <= t <= total_sec})


def make_chunks(bounds, total_sec, speakers, active, words):
    """One dict per chunk: its times, speaker and words."""
    per_chunk = [[] for _ in bounds[:-1]]
    if words:
        mids = np.array([(w["start"] + w["end"]) / 2 for w in words])
        owner = np.clip(np.searchsorted(bounds, mids, side="right") - 1, 0, len(per_chunk) - 1)
        for w, k in zip(words, owner):
            per_chunk[k].append(w)
    return [{
        "start": round(s, 3), "end": round(e, 3),
        "speaker": speaker_of(s, e, speakers, active),
        "words": mine,
    } for (s, e), mine in zip(zip(bounds, bounds[1:]), per_chunk)]


def export(audio_path, chunks, out_dir):
    """Cut the wavs from the original audio (mono, original sample rate). Returns (file, duration) per chunk."""
    wav_dir = out_dir / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)
    for old in wav_dir.glob("*.wav"):  # an earlier run's wavs
        old.unlink()
    files = []
    with sf.SoundFile(audio_path) as f:
        sr = f.samplerate
        fade = np.linspace(0.0, 1.0, max(1, round(FADE_SEC * sr)), dtype=np.float32)
        for k, c in enumerate(chunks):
            a = round(c["start"] * sr)
            b = f.frames if k == len(chunks) - 1 else min(round(c["end"] * sr), f.frames)  # the last wav ends at the last sample
            f.seek(a)
            x = f.read(b - a, dtype="float32", always_2d=True).mean(axis=1)
            m = min(len(fade), len(x) // 2)
            if m:
                x[:m] *= fade[:m]
                x[-m:] *= fade[:m][::-1]
            name = f"{k:04d}_{c['speaker'] or 'none'}.wav"
            sf.write(wav_dir / name, x, sr, subtype="PCM_16")
            files.append((f"wavs/{name}", round(len(x) / sr, 3)))
    return files


def check(total_sec, chunks, files, words):
    """Raises if anything was lost: the chunks must cover the whole audio and hold every word, in order."""
    covered = sum(c["end"] - c["start"] for c in chunks)
    wav_sec = sum(d for _, d in files)
    assert chunks[0]["start"] == 0 and abs(chunks[-1]["end"] - total_sec) < 0.002, "chunks don't span the audio"
    assert all(a["end"] == b["start"] for a, b in zip(chunks, chunks[1:])), "gap or overlap between chunks"
    assert abs(covered - total_sec) < 0.01, f"chunks cover {covered:.2f}s of {total_sec:.2f}s"
    assert abs(wav_sec - total_sec) < 0.05, f"wavs hold {wav_sec:.2f}s, the audio {total_sec:.2f}s"
    assert max(d for _, d in files) <= MAX_CHUNK_SEC + 0.01, "a wav is longer than MAX_CHUNK_SEC"
    if words is not None:
        assert [w for c in chunks for w in c["words"]] == words, "words lost or out of order"


def process(audio_path):
    name = audio_path.stem
    rttm_path = out_root / "diarization" / name / "nemotron" / f"{name}.rttm"
    out_dir = out_root / "chunks" / name
    json_path = out_dir / "chunks.json"
    if json_path.exists() and not FORCE:
        print("  done before, skipping")
        return
    if not rttm_path.exists():
        raise FileNotFoundError(f"no RTTM: run 04_diarization on {audio_path.name} first")
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in ("chunks.json", "manifest.jsonl", "summary.json"):  # this or the earlier layout
        (out_dir / old).unlink(missing_ok=True)

    words = None
    words_path = words_dir / f"{name}.json"
    if words_path.exists():
        words = json.loads(words_path.read_text(encoding="utf-8"))["words"]
    else:
        print("  no alignment in data/alignments: chunks without text")

    total_sec = sf.info(str(audio_path)).duration
    speakers, active, labels = frame_labels(read_rttm(rttm_path), math.ceil(total_sec * RES))
    bounds = chunk_bounds(labels, total_sec, words or [], speakers, active)
    chunks = make_chunks(bounds, total_sec, speakers, active, words)
    files = export(audio_path, chunks, out_dir)
    check(total_sec, chunks, files, words)

    record = {
        "audio": audio_path.name,
        "duration": round(total_sec, 3),
        "chunks": [{
            "file": file,
            "speaker": c["speaker"],
            "start": c["start"],
            "end": c["end"],
            "duration": duration,
            "text": " ".join(w["word"] for w in c["words"]) if words is not None else None,
        } for c, (file, duration) in zip(chunks, files)],
    }
    tmp = json_path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, json_path)  # written last = done marker

    lengths = [d for _, d in files]
    print(f"  {len(chunks)} chunks ({sum(c['speaker'] is None for c in chunks)} without speech), "
          f"wavs {sum(lengths) / 60:.1f} min for {total_sec / 60:.1f} min of audio, "
          f"median {np.median(lengths):.1f}s, max {max(lengths):.1f}s")

## Run

In [16]:
selected = wavs if ONLY is None else [wavs[i] for i in ONLY]
failed = {}

for n, audio_path in enumerate(selected, 1):
    print(f"[{n}/{len(selected)}] {audio_path.name}")
    try:
        process(audio_path)
    except Exception:
        failed[audio_path.name] = traceback.format_exc()
        print("  FAILED (continuing)")

print(f"\ndone: {len(selected) - len(failed)}, failed: {len(failed)}")
for name, tb in failed.items():
    print("\n" + name + "\n" + tb)

[1/31] 18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak.wav
  done before, skipping
[2/31] 2026-yilda Google'ga ishga kirish XARITASI.wav
  done before, skipping
[3/31] 28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili.wav
  done before, skipping
[4/31] 29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi.wav
  done before, skipping
[5/31] 31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda.wav
  done before, skipping
[6/31] 40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son.wav
  done before, skipping
[7/31] 50 yoshida ham studentlikni tanlaydigan odam bilan suhbat.wav
  done before, skipping
[8/31] 99% odamlar buni eplay olmaydi - Iqtibos podcast #27.wav
  done before, skipping
[9/31] AQLI PASTlar avlodi (bo'lib qolyapmiz).wav
  done before, skipping
[10/31] Afsus qilmaslik uchun bugun nima qilish kerak - Hasanxon Yahyo Abdulmajid.wav


## Every audio
Read from the `chunks.json` files. `covered` and `wavs` must both equal `audio` (nothing dropped, nothing doubled).

In [17]:
records = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((out_root / "chunks").glob("*/chunks.json"))]

print(f"{'audio':50} {'audio min':>9} {'covered':>8} {'wavs':>7} {'chunks':>7} {'no speech':>9} "
      f"{'< 2 s':>6} {'median s':>9} {'max s':>6}")
for r in records:
    c = r["chunks"]
    lengths = [x["duration"] for x in c]
    print(f"{r['audio'][:50]:50} {r['duration'] / 60:9.1f} {sum(x['end'] - x['start'] for x in c) / 60:8.1f} "
          f"{sum(lengths) / 60:7.1f} {len(c):7d} {sum(x['speaker'] is None for x in c):9d} "
          f"{sum(d < 2 for d in lengths):6d} {np.median(lengths):9.1f} {max(lengths):6.1f}")
all_c = [x for r in records for x in r["chunks"]]
print(f"\n{len(records)} audios: {sum(r['duration'] for r in records) / 3600:.2f} h of audio, "
      f"{sum(x['duration'] for x in all_c) / 3600:.2f} h of wavs, {len(all_c)} chunks "
      f"({sum(x['speaker'] is None for x in all_c)} without speech)")

audio                                              audio min  covered    wavs  chunks no speech  < 2 s  median s  max s
18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingi     146.8    146.8   154.5    1557        13    642       2.7   29.7
2026-yilda Google'ga ishga kirish XARITASI.wav          30.7     30.7    31.1      80         1      0      25.7   29.8
28-son - GTA VI, Fransiyadagi noroziliklar, Tesla       39.5     39.5    40.1     131         8     24      26.0   29.9
29-son - Krinjdan tortib siyosatgacha hafta davomi      40.9     40.9    41.5     115        13      2      28.2   29.9
31 - son - Birinçi besoqol, musulmon, tub amerikal      47.8     47.8    48.5     135        12      5      27.7   29.9
40 daqiqada yillar davomida yo'qotganimizni qaytar      44.0     44.0    44.5      96         0      0      29.4   29.9
50 yoshida ham studentlikni tanlaydigan odam bilan     120.0    120.0   124.7     935         8    315       4.0   29.7
99% odamlar buni eplay olmaydi - Iqtibos

## Listen to a few chunks

In [18]:
from IPython.display import Audio, display

i = 0       # index from the list at the top
first = 0   # first chunk to play
out_dir = out_root / "chunks" / wavs[i].stem
record = json.loads((out_dir / "chunks.json").read_text(encoding="utf-8"))
for c in record["chunks"][first:first + 5]:
    print(f"{c['start']:8.2f}-{c['end']:8.2f}s  {c['speaker']}  {c['text']}")
    display(Audio(str(out_dir / c["file"])))

    0.00-    3.36s  speaker_0  Nechta odam sizga shogird bo'laman deb keladi? Imkoniyat berasiz, nechtasi qoladi?


    3.36-    5.12s  speaker_1  Kelishini bilmadim-u, yozadi lekin ko'pchilik.


    5.12-    6.64s  speaker_0  Xo'p, nechtasi yozadi va nechtasi


    6.64-    9.21s  speaker_1  keladi? Kirib keladiganlarni hammasini oldik, lekin bittasidan tashqari.


    9.21-   29.36s  speaker_0  Yozish bilan kelishni orasida ham farq bor-da. Endi xo'p, omad, omad nima? O'zi bor narsami? Menimcha yo'q narsa. Omad deb o'ylagan narsamiz faqatgina biz o'sha imkoniyatni tushunib yetganligimiz o'sha vaqtda ekan. Chunki imkoniyat har doim bor. 10 soat davomida nechta podkastni ko'rish mumkin? Tekindan-tekin. Ishga boraman, bir
